# Phase 1 — InternVL2-2B Fine-tuning with LLaMA-Factory

**Project:** Infrastructure inspection VLM  
**Model:** InternVL2-2B  
**Method:** QLoRA via LLaMA-Factory  
**Dataset:** 214 ITSEOA inspection pairs (image + French commentary)  
**Hardware:** Google Colab T4 GPU

---

### Why LLaMA-Factory instead of raw HuggingFace Trainer?

InternVL2 has a custom architecture — it is not a standard HuggingFace causal LM.
It has its own image preprocessing pipeline, its own image token format (`<IMG_CONTEXT>`),
and its own `forward()` signature that the standard Trainer does not understand.

LLaMA-Factory is an open-source fine-tuning framework that has **native, tested support
for InternVL2**. The Shanghai AI Lab team (who built InternVL2) recommend it.
It handles all the internals correctly so we can focus on our data and hyperparameters.

### What this notebook does
1. Sets up the environment and mounts Google Drive
2. Installs LLaMA-Factory
3. Loads data from MongoDB (or JSON fallback)
4. Converts our dataset to the ShareGPT format LLaMA-Factory expects
5. Writes a YAML config file that defines the training run
6. Runs training with a single command
7. Evaluates the fine-tuned model
8. Saves everything to Google Drive

### How to resume after a Colab disconnect
Run Sections 0 and 1 only — then jump to Section 5 and add `resume_from_checkpoint: true` to the YAML.

---
## Section 0 — Environment Setup

### What we are doing
We check the GPU, mount Google Drive, and install LLaMA-Factory.

### Why clone LLaMA-Factory into Drive?
If Colab disconnects and restarts, we do not need to reinstall — it is already on Drive.
We also install it as a Python package so `llamafactory-cli` is available as a command.

### What is LLaMA-Factory?
An open-source framework by Hiyouga that supports fine-tuning 100+ LLMs and VLMs
including InternVL2, LLaMA, Qwen, Mistral, and many others. It abstracts all
model-specific complexity and exposes a clean YAML configuration interface.
GitHub: https://github.com/hiyouga/LLaMA-Factory

In [ ]:
# ── Check GPU ─────────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────
# Everything will be saved under /content/drive/MyDrive/CID/
# This survives Colab session resets.

from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive mounted')

In [ ]:
# ── Install LLaMA-Factory ─────────────────────────────────────────────────
import os

DRIVE_BASE = '/content/drive/MyDrive/CID'
LF_DIR     = f'{DRIVE_BASE}/LLaMA-Factory'

os.makedirs(DRIVE_BASE, exist_ok=True)

if not os.path.exists(LF_DIR):
    print('Cloning LLaMA-Factory into Drive (first time only)...')
    os.system(f'git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git {LF_DIR}')
else:
    print('LLaMA-Factory already cloned, pulling latest...')
    os.system(f'cd {LF_DIR} && git pull')

# Install as a package — this registers the llamafactory-cli command
os.system(f'pip install -q -e "{LF_DIR}[torch,metrics]"')
os.system('pip install -q bitsandbytes pymongo scikit-learn')

print('\n✓ LLaMA-Factory installed')

---
## Section 1 — Configuration

### What we are doing
All paths, hyperparameters, and settings in one place.
This is the only cell you need to edit to control the training run.

### Hyperparameter explanations
- **lora_rank (16):** how many parameters LoRA adds. Higher = more capacity but more VRAM. 16 is the standard starting point for domain adaptation.
- **lora_alpha (32):** scaling factor. Convention is 2× the rank. Controls how strongly the LoRA updates influence the model.
- **lora_dropout (0.05):** 5% of LoRA weights randomly zeroed during training to prevent overfitting on our small dataset.
- **num_train_epochs (3):** 3 passes through 214 samples. More than 3 risks the model memorizing rather than generalizing.
- **per_device_train_batch_size (2):** 2 samples per GPU step — maximum that fits on T4 with QLoRA.
- **gradient_accumulation_steps (8):** accumulate gradients over 8 steps before updating weights. Effective batch size = 2×8 = 16. This stabilizes training without needing more memory.
- **learning_rate (2e-4):** how fast the model updates its weights. Standard QLoRA starting point from the original paper.
- **cutoff_len (2048):** maximum token length. Our commentaries are long so we need 2048.
- **freeze_vision_tower:** keeps the image encoder frozen. With only 214 samples, training the vision encoder would cause overfitting. The encoder already knows how to extract visual features — we only teach the language side.

In [ ]:
# ── Configuration — edit this cell to control the training ────────────────
from google.colab import userdata
import os

# ── MongoDB ────────────────────────────────────────────────────────────────
# Add MONGO_URI to Colab Secrets: click the 🔑 icon in the left sidebar
# Key: MONGO_URI   Value: your Atlas URI
try:
    MONGO_URI = userdata.get('MONGO_URI')
except:
    MONGO_URI = ''

MONGO_DB   = 'inspection_db'
MONGO_COLL = 'itseoa_pairs'

# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_BASE    = '/content/drive/MyDrive/CID'
LF_DIR        = f'{DRIVE_BASE}/LLaMA-Factory'
PHASE1_DIR    = f'{DRIVE_BASE}/phase1'
DATA_DIR      = f'{LF_DIR}/data'
OUTPUT_DIR    = f'{PHASE1_DIR}/output'
DATASET_NAME  = 'cid_itseoa'
JSON_FALLBACK = f'{DRIVE_BASE}/data/finetuning_dataset.json'

# ── Model & Training ───────────────────────────────────────────────────────
MODEL_NAME   = 'OpenGVLab/InternVL2-2B'
LORA_RANK    = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
NUM_EPOCHS   = 3
BATCH_SIZE   = 2
GRAD_ACCUM   = 8
LR           = '2e-4'
CUTOFF_LEN   = 2048
WARMUP_RATIO = 0.03
VAL_SPLIT    = 0.1
RANDOM_SEED  = 42
SAVE_STEPS   = 50

for d in [PHASE1_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print('✓ Configuration loaded')
print(f'  Model:           {MODEL_NAME}')
print(f'  MongoDB:         {"connected" if MONGO_URI else "not set — will use JSON"}')
print(f'  Effective batch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'  Output:          {OUTPUT_DIR}')

---
## Section 2 — Data Loading & Conversion

### What we are doing
We load our 214 pairs from MongoDB (or JSON) and convert them to **ShareGPT format**,
which is what LLaMA-Factory expects.

### What is ShareGPT format?
ShareGPT is a simple JSON format originally used to share ChatGPT conversations.
Each sample is a conversation with a `messages` list and an `images` list:

```json
{
  "messages": [
    {"role": "user",      "content": "<image>\nAnalyse cette image..."},
    {"role": "assistant", "content": "Le désordre observé est..."}
  ],
  "images": ["/path/to/image.jpg"]
}
```

The `<image>` token in the user message tells LLaMA-Factory where to inject the visual features.
LLaMA-Factory loads the image from the path and handles all preprocessing for InternVL2 internally —
this is the part that was causing all the errors in our previous approach.

### Why save images as files?
LLaMA-Factory's data pipeline reads images from disk, not from base64 strings.
This is fine since images will be on Google Drive and persist across sessions.

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────
import json, base64
from io import BytesIO
from PIL import Image

def load_from_mongo(uri, db, coll):
    from pymongo import MongoClient
    client = MongoClient(uri, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    records = list(client[db][coll].find({}))
    print(f'✓ Loaded {len(records)} records from MongoDB')
    return records

def load_from_json(path):
    with open(path, encoding='utf-8') as f:
        records = json.load(f)
    print(f'✓ Loaded {len(records)} records from JSON')
    return records

records = None
if MONGO_URI:
    try:
        records = load_from_mongo(MONGO_URI, MONGO_DB, MONGO_COLL)
    except Exception as e:
        print(f'MongoDB failed ({e}) — falling back to JSON')

if records is None:
    records = load_from_json(JSON_FALLBACK)

print(f'Total: {len(records)} records')

In [ ]:
# ── Convert to ShareGPT format ────────────────────────────────────────────
import os
from sklearn.model_selection import train_test_split

IMAGES_DIR = f'{DATA_DIR}/cid_images'
os.makedirs(IMAGES_DIR, exist_ok=True)

INSTRUCTION = (
    "Tu es un ingénieur civil spécialisé dans l'inspection des ouvrages d'art. "
    "Analyse cette image et rédige un commentaire d'inspection professionnel "
    "couvrant : le désordre observé, ses causes probables, sa gravité, "
    "et les suites à donner. Ne décris que ce qui est visuellement observable."
)

sharegpt_data = []
skipped       = 0

for i, r in enumerate(records):
    meta       = r.get('meta', {})
    commentary = r.get('commentary_full') or r.get('response', '')
    image_b64  = r.get('image_b64', '')

    if not image_b64 or not commentary:
        skipped += 1
        continue

    # Save image to disk
    img_filename = f'sample_{i:04d}.jpg'
    img_path     = f'{IMAGES_DIR}/{img_filename}'

    if not os.path.exists(img_path):
        img_bytes = base64.b64decode(image_b64)
        img = Image.open(BytesIO(img_bytes)).convert('RGB')
        img.save(img_path, 'JPEG', quality=95)

    # Build ShareGPT sample
    # <image> is the placeholder that LLaMA-Factory replaces with visual features
    sharegpt_data.append({
        'messages': [
            {'role': 'user',      'content': f'<image>\n{INSTRUCTION}'},
            {'role': 'assistant', 'content': commentary}
        ],
        'images': [img_path]
    })

print(f'✓ Converted {len(sharegpt_data)} samples ({skipped} skipped)')

# Preview
s = sharegpt_data[0]
print(f'\nSample preview:')
print(f'  Image:     {s["images"][0]}')
print(f'  Assistant: {s["messages"][1]["content"][:120]}...')

In [ ]:
# ── Train / validation split & save ──────────────────────────────────────
# We split before saving so train and val are separate JSON files.
# With 214 samples and VAL_SPLIT=0.1: ~193 train, ~21 val.

train_data, val_data = train_test_split(
    sharegpt_data,
    test_size    = VAL_SPLIT,
    random_state = RANDOM_SEED,
    shuffle      = True,
)

print(f'✓ Train: {len(train_data)} | Val: {len(val_data)}')

TRAIN_FILE = f'{DATA_DIR}/{DATASET_NAME}_train.json'
VAL_FILE   = f'{DATA_DIR}/{DATASET_NAME}_val.json'

with open(TRAIN_FILE, 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open(VAL_FILE, 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

print(f'  Saved: {TRAIN_FILE}')
print(f'  Saved: {VAL_FILE}')

In [ ]:
# ── Register dataset in LLaMA-Factory ────────────────────────────────────
# LLaMA-Factory keeps a registry at data/dataset_info.json.
# We add entries for our train and val sets.
# 'formatting: sharegpt' tells it which parser to use.
# 'columns' maps our field names to the expected keys.

dataset_info_path = f'{DATA_DIR}/dataset_info.json'

if os.path.exists(dataset_info_path):
    with open(dataset_info_path) as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

dataset_info[f'{DATASET_NAME}_train'] = {
    'file_name' : f'{DATASET_NAME}_train.json',
    'formatting': 'sharegpt',
    'columns'   : {'messages': 'messages', 'images': 'images'}
}

dataset_info[f'{DATASET_NAME}_val'] = {
    'file_name' : f'{DATASET_NAME}_val.json',
    'formatting': 'sharegpt',
    'columns'   : {'messages': 'messages', 'images': 'images'}
}

with open(dataset_info_path, 'w') as f:
    json.dump(dataset_info, f, indent=2)

print(f'✓ Dataset registered: {DATASET_NAME}_train / {DATASET_NAME}_val')

---
## Section 3 — Training Configuration (YAML)

### What we are doing
We generate the YAML config file that tells LLaMA-Factory everything about the training run.

### Why a YAML file?
LLaMA-Factory's CLI takes a YAML config as input. This is better than dozens of
command-line flags — it is readable, version-controllable, and easy to share with your boss.
We generate it from Python so we can use our config variables directly.

### Key settings

**`stage: sft`** — Supervised Fine-Tuning. We show the model (image + prompt, commentary)
pairs and train it to produce the commentary. No reinforcement learning needed.

**`finetuning_type: lora`** + **`quantization_bit: 4`** = QLoRA. Load model in 4-bit,
add trainable LoRA matrices on top. VRAM usage drops from ~16GB to ~5GB.

**`lora_target: all`** — apply LoRA to all linear layers. LLaMA-Factory knows which
layers are appropriate for InternVL2 when `all` is used. Safer than listing names manually.

**`freeze_vision_tower: true`** — keep the image encoder frozen. It already knows how
to extract visual features from photos. We only need to teach the language side
how to describe infrastructure defects in French using the ITSEOA vocabulary.

**`template: intern_vl`** — tells LLaMA-Factory to use InternVL2's specific conversation
template and image preprocessing. This is what was failing in our previous approach.
LLaMA-Factory handles the `<IMG_CONTEXT>` tokens, tiling, and normalization internally.

In [ ]:
# ── Generate YAML training config ────────────────────────────────────────

yaml_content = f"""### Model
model_name_or_path: {MODEL_NAME}
trust_remote_code: true

### Training stage
stage: sft
do_train: true
finetuning_type: lora

### QLoRA: load model in 4-bit to fit on T4
quantization_bit: 4
quantization_type: nf4

### LoRA adapter config
lora_rank: {LORA_RANK}
lora_alpha: {LORA_ALPHA}
lora_dropout: {LORA_DROPOUT}
lora_target: all

### Vision encoder: keep frozen (too few samples to train it)
freeze_vision_tower: true

### Dataset
dataset: {DATASET_NAME}_train
eval_dataset: {DATASET_NAME}_val
template: intern_vl
cutoff_len: {CUTOFF_LEN}
val_size: 0

### Output
output_dir: {OUTPUT_DIR}
logging_dir: {OUTPUT_DIR}/logs
save_steps: {SAVE_STEPS}
save_total_limit: 3
logging_steps: 10

### Training hyperparameters
per_device_train_batch_size: {BATCH_SIZE}
per_device_eval_batch_size: 1
gradient_accumulation_steps: {GRAD_ACCUM}
num_train_epochs: {NUM_EPOCHS}
learning_rate: {LR}
lr_scheduler_type: cosine
warmup_ratio: {WARMUP_RATIO}
max_grad_norm: 0.3
weight_decay: 0.001

### Precision: bfloat16 is faster and more stable than float16 on T4
bf16: true

### Evaluation
evaluation_strategy: steps
eval_steps: {SAVE_STEPS}
load_best_model_at_end: true
metric_for_best_model: eval_loss

### Misc
seed: {RANDOM_SEED}
report_to: none
overwrite_output_dir: true
dataloader_pin_memory: false
"""

YAML_PATH = f'{PHASE1_DIR}/train_config.yaml'
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print(f'✓ Config saved to {YAML_PATH}')

---
## Section 4 — Verify Setup

### What we are doing
Before launching training (which takes ~1 hour), we verify that:
- All images are on disk
- The dataset JSON files are readable  
- The YAML config references the correct paths

This catches errors in 30 seconds instead of wasting an hour of GPU time.

In [ ]:
# ── Pre-flight checks ────────────────────────────────────────────────────
import json, os

errors = []

# Check train file
with open(TRAIN_FILE) as f:
    train_check = json.load(f)
print(f'✓ Train file: {len(train_check)} samples')

# Check val file
with open(VAL_FILE) as f:
    val_check = json.load(f)
print(f'✓ Val file:   {len(val_check)} samples')

# Check images exist
missing = [s for s in train_check + val_check if not os.path.exists(s['images'][0])]
if missing:
    errors.append(f'{len(missing)} images missing from disk — re-run the conversion cell')
else:
    print(f'✓ All {len(train_check)+len(val_check)} images found on disk')

# Check YAML file
if not os.path.exists(YAML_PATH):
    errors.append('YAML config not found — re-run Section 3')
else:
    print(f'✓ YAML config found')

# Check LLaMA-Factory installed
import subprocess
result = subprocess.run(['llamafactory-cli', 'version'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'✓ LLaMA-Factory: {result.stdout.strip()}')
else:
    errors.append('llamafactory-cli not found — re-run the install cell')

if errors:
    print('\n⚠ ERRORS FOUND:')
    for e in errors: print(f'  - {e}')
else:
    print('\n✓ All checks passed — ready to train!')

---
## Section 5 — Training

### What we are doing
We launch training with a single command: `llamafactory-cli train config.yaml`

### What happens internally
1. LLaMA-Factory loads InternVL2-2B in 4-bit quantization (~4GB download first time)
2. It applies QLoRA adapters to all linear layers in the language model
3. For each training step:
   - Images are loaded from disk and processed through InternVL2's official pipeline:
     resize to 448×448, split into tiles, normalize with ImageNet stats
   - Each image becomes 256 `<IMG_CONTEXT>` tokens via the vision encoder
   - Visual tokens + text tokens are fed to the language model together
   - The model predicts the next token in the commentary
   - Loss is computed only on assistant response tokens (not the prompt — we mask those with -100)
   - Gradients flow only through the language model LoRA layers (vision encoder stays frozen)
   - LoRA matrices A and B are updated by the optimizer
4. Every 50 steps: checkpoint saved to Drive, validation loss computed
5. At the end: the checkpoint with the lowest validation loss is kept

### What to watch in the output
- **`loss`** should decrease over time — ideally from ~3.0 down to ~1.0
- **`eval_loss`** should also decrease — if it goes up while `loss` goes down, overfitting is happening
- **`epoch`** shows progress through the 3 epochs

### Expected runtime: 60–90 minutes on T4

In [ ]:
# ── Launch training ───────────────────────────────────────────────────────
# This single command handles everything.
# To resume after a disconnect: add  resume_from_checkpoint: true  to the YAML
# then re-run this cell.

import os
os.system(f'cd {LF_DIR} && llamafactory-cli train {YAML_PATH}')

---
## Section 6 — Evaluation

### What we are doing
We generate commentaries for the 21 validation images using the fine-tuned model
and compare them visually to the expert-written ground truth.

### What is a good result with 214 training samples?
Do not expect perfect output. Good signs:
- Model uses French technical vocabulary: fissures, éclatements, armatures, ségrégation...
- It correctly identifies the defect type visible in the image
- The commentary follows the ITSEOA structure: description → causes → gravité → suites
- Each image produces a different commentary (not the same text for everything)

Bad signs (overfitting):
- Every image produces the same or very similar commentary
- The model copies entire sentences from the training data word for word

In [ ]:
# ── Load fine-tuned model ────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import PeftModel
import json
from PIL import Image
import textwrap
from IPython.display import display

# Load val data
with open(VAL_FILE, encoding='utf-8') as f:
    val_samples = json.load(f)

print(f'✓ {len(val_samples)} validation samples')

# Load model + adapters
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

print('Loading base model...')
base_model = AutoModel.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb,
    device_map          = 'auto',
    trust_remote_code   = True,
)

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()
print('✓ Model loaded')

In [ ]:
# ── Generate predictions for 5 validation samples ────────────────────────
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
import numpy as np

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.Resize((448, 448), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

NUM_DISPLAY = 5

print('=' * 70)
print('EVALUATION — Fine-tuned Model vs Ground Truth')
print('=' * 70)

for i, sample in enumerate(val_samples[:NUM_DISPLAY]):
    img_path     = sample['images'][0]
    ground_truth = sample['messages'][1]['content']

    print(f'\n── Sample {i+1}/{NUM_DISPLAY} ──────────────────────────────────────')

    img = Image.open(img_path).convert('RGB')
    thumb = img.copy()
    thumb.thumbnail((350, 350))
    display(thumb)

    # Preprocess image
    pixel_values = transform(img).unsqueeze(0).to(model.device, dtype=torch.float16)

    # Generate using InternVL2's chat API
    with torch.no_grad():
        response = model.chat(
            tokenizer        = tokenizer,
            pixel_values     = pixel_values,
            question         = INSTRUCTION,
            generation_config= dict(max_new_tokens=400, do_sample=False),
        )

    print('\n  GENERATED:')
    print(textwrap.fill(response[:500], width=68, initial_indent='  ', subsequent_indent='  '))
    print('\n  GROUND TRUTH:')
    print(textwrap.fill(ground_truth[:500], width=68, initial_indent='  ', subsequent_indent='  '))
    print()

---
## Section 7 — Export Merged Model

### What we are doing
We merge the LoRA adapter weights into the base model and save the result to Drive.

### Why merge?
During training, we have two separate things:
- The frozen base model (InternVL2-2B, ~4GB)
- The LoRA adapters (~100MB of trainable matrices)

At inference, the output is: `W_base + (alpha/rank) × A × B`

Merging computes this addition permanently, producing a single model file.
The result is identical in quality but has no PEFT dependency — simpler for Phase 2.

### What `llamafactory-cli export` does
It loads the base model in float16 (not quantized — you can't merge into quantized weights),
loads the LoRA adapters, calls `merge_and_unload()`, and saves the result.
The tokenizer and processor config are included so the model is fully self-contained.

In [ ]:
# ── Export merged model ──────────────────────────────────────────────────
import os

MERGED_DIR = f'{PHASE1_DIR}/merged_model'
os.makedirs(MERGED_DIR, exist_ok=True)

export_yaml = f"""model_name_or_path: {MODEL_NAME}
adapter_name_or_path: {OUTPUT_DIR}
template: intern_vl
finetuning_type: lora
trust_remote_code: true
export_dir: {MERGED_DIR}
export_size: 4
export_device: cpu
export_legacy_format: false
"""

EXPORT_YAML = f'{PHASE1_DIR}/export_config.yaml'
with open(EXPORT_YAML, 'w') as f:
    f.write(export_yaml)

print('Merging LoRA into base model (~5 minutes)...')
os.system(f'cd {LF_DIR} && llamafactory-cli export {EXPORT_YAML}')
print(f'\n✓ Merged model saved to {MERGED_DIR}')

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────
print('=' * 60)
print('PHASE 1 COMPLETE')
print('=' * 60)
print(f'Model:             {MODEL_NAME}')
print(f'Method:            QLoRA (rank={LORA_RANK}, alpha={LORA_ALPHA})')
print(f'Training samples:  {len(train_data)}')
print(f'Validation samples:{len(val_data)}')
print(f'Epochs:            {NUM_EPOCHS}')
print()
print('Saved to Google Drive:')
print(f'  Checkpoints: {OUTPUT_DIR}')
print(f'  Merged model:{MERGED_DIR}')
print(f'  YAML config: {YAML_PATH}')
print()
print('Next steps:')
print('  1. Review the evaluation outputs in Section 6')
print('  2. Once GYU-DET and field reports are added, retrain with full dataset')
print('  3. Use the merged model in Phase 2 (LangGraph pipeline)')